In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import ast
import re
from tqdm import tqdm

# Load results
results = pd.read_csv("/Users/reddy/AWW_and_ICU/pgfgleam/pgfgleam/datasets/full_results.csv")

# Parse vector columns with Julia-safe parser - UPDATED FOR 8 WW CONFIGS
vector_cols = [
    'ICU_detection_times', 'ICU_local_cases_samples',
    # Base p_det = 0.04
    'WW_008_10pct_detection_times', 'WW_008_10pct_local_cases_samples',
    'WW_008_25pct_detection_times', 'WW_008_25pct_local_cases_samples',
    'WW_008_50pct_detection_times', 'WW_008_50pct_local_cases_samples',
    'WW_008_100pct_detection_times', 'WW_008_100pct_local_cases_samples',
    # Base p_det = 0.16
    'WW_016_10pct_detection_times', 'WW_016_10pct_local_cases_samples',
    'WW_016_25pct_detection_times', 'WW_016_25pct_local_cases_samples',
    'WW_016_50pct_detection_times', 'WW_016_50pct_local_cases_samples',
    'WW_016_100pct_detection_times', 'WW_016_100pct_local_cases_samples'
]

def parse_julia_vector(s):
    """
    Parse Julia vector strings ie. 'Float64[]'
    """
    if pd.isna(s):
        return []
    s = str(s).strip()
    # Handle Julia empty vector notation: Float64[], Int64[], etc.
    if re.match(r'^[A-Za-z0-9]+\[\]$', s):
        return []
    # Handle standard array notation
    try:
        return ast.literal_eval(s)
    except:
        # If all else fails, try to extract numbers
        try:
            numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
            return [float(n) for n in numbers]
        except:
            return []

for col in vector_cols:
    print(f"Parsing {col}...")
    results[col] = results[col].apply(parse_julia_vector)

print(f"Loaded {len(results)} results")

def hierarchical_bootstrap_ci(subset, sample_col, n_bootstrap=100000):
    """
    Bootstrap CI accounting for both within-country and between-country uncertainty.
    Works for both detection times and local cases.
    
    Process:
    1. Sample a country uniformly at random
    2. Sample one value from that country's samples
    3. Repeat n_bootstrap times
    4. Calculate percentiles
    """
    bootstrap_samples = []
    
    # Get valid countries (those with at least one sample)
    valid_countries = []
    country_samples = []
    
    for idx, row in subset.iterrows():
        samples = row[sample_col]
        valid_samples = [s for s in samples if pd.notna(s) and not np.isnan(s) and np.isfinite(s)]
        if len(valid_samples) > 0:
            valid_countries.append(idx)
            country_samples.append(valid_samples)
    
    if len(valid_countries) == 0:
        return {'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}
    
    # Bootstrap the two-stage process
    for _ in range(n_bootstrap):
        # Stage 1: Pick a country uniformly at random
        country_idx = np.random.choice(len(valid_countries))
        # Stage 2: Pick one sample from that country
        value = np.random.choice(country_samples[country_idx])
        bootstrap_samples.append(value)
    
    # Calculate statistics
    mean_val = np.mean(bootstrap_samples)
    ci_lower = np.percentile(bootstrap_samples, 2.5)
    ci_upper = np.percentile(bootstrap_samples, 97.5)
    
    return {'mean': mean_val, 'ci_lower': ci_lower, 'ci_upper': ci_upper}

def simple_mean_ci(subset, mean_col):
    """
    Simple CI of country means (current approach for comparison)
    """
    valid_values = [v for v in subset[mean_col] if pd.notna(v) and not np.isnan(v) and np.isfinite(v)]
    
    if len(valid_values) == 0:
        return {'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan}
    
    mean_val = np.mean(valid_values)
    se = stats.sem(valid_values)
    ci_lower = mean_val - 1.96 * se
    ci_upper = mean_val + 1.96 * se
    
    return {'mean': mean_val, 'ci_lower': ci_lower, 'ci_upper': ci_upper}

# Get unique combinations
combinations = results[['R0', 'gen_time']].drop_duplicates().sort_values(['R0', 'gen_time'])

print("\n" + "="*100)
print("HIERARCHICAL BOOTSTRAP CIs vs SIMPLE CIs (8 WW CONFIGS + ICU)")
print("="*100)

summary_data = []

for _, combo in tqdm(list(combinations.iterrows()), desc="Processing combinations"):
    R0_val = combo['R0']
    gen_time_val = combo['gen_time']
    
    subset = results[(results['R0'] == R0_val) & (results['gen_time'] == gen_time_val)]
    
    print(f"\n{'-'*100}")
    print(f"R0 = {R0_val}, Generation Time = {gen_time_val} days")
    print(f"Number of countries: {len(subset)}")
    print(f"{'-'*100}")
    
    # Dictionary to store all results
    result_dict = {
        'R0': R0_val,
        'gen_time': gen_time_val,
        'n_countries': len(subset)
    }
    
    # ========================================================================
    # ICU DETECTION TIME & CASES
    # ========================================================================
    print("\n" + "="*50)
    print("ICU")
    print("="*50)
    
    icu_time_simple = simple_mean_ci(subset, 'ICU_mean_detection_time')
    icu_time_hierarchical = hierarchical_bootstrap_ci(subset, 'ICU_detection_times')
    print(f"Detection Time - Simple: {icu_time_simple['mean']:.1f}: ({icu_time_simple['ci_lower']:.1f}, {icu_time_simple['ci_upper']:.1f})")
    print(f"Detection Time - Hierarchical: {icu_time_hierarchical['mean']:.1f}: ({icu_time_hierarchical['ci_lower']:.1f}, {icu_time_hierarchical['ci_upper']:.1f})")
    
    icu_cases_simple = simple_mean_ci(subset, 'ICU_mean_local_cases')
    icu_cases_hierarchical = hierarchical_bootstrap_ci(subset, 'ICU_local_cases_samples')
    print(f"Local Cases - Simple: {icu_cases_simple['mean']:.1f}: ({icu_cases_simple['ci_lower']:.1f}, {icu_cases_simple['ci_upper']:.1f})")
    print(f"Local Cases - Hierarchical: {icu_cases_hierarchical['mean']:.1f}: ({icu_cases_hierarchical['ci_lower']:.1f}, {icu_cases_hierarchical['ci_upper']:.1f})")
    
    result_dict.update({
        'ICU_time_simple_mean': icu_time_simple['mean'],
        'ICU_time_simple_ci_lower': icu_time_simple['ci_lower'],
        'ICU_time_simple_ci_upper': icu_time_simple['ci_upper'],
        'ICU_time_hierarchical_mean': icu_time_hierarchical['mean'],
        'ICU_time_hierarchical_pi_lower': icu_time_hierarchical['ci_lower'],
        'ICU_time_hierarchical_pi_upper': icu_time_hierarchical['ci_upper'],
        'ICU_cases_simple_mean': icu_cases_simple['mean'],
        'ICU_cases_simple_ci_lower': icu_cases_simple['ci_lower'],
        'ICU_cases_simple_ci_upper': icu_cases_simple['ci_upper'],
        'ICU_cases_hierarchical_mean': icu_cases_hierarchical['mean'],
        'ICU_cases_hierarchical_pi_lower': icu_cases_hierarchical['ci_lower'],
        'ICU_cases_hierarchical_pi_upper': icu_cases_hierarchical['ci_upper']
    })
    
    # ========================================================================
    # ALL 8 WW CONFIGURATIONS
    # ========================================================================
    ww_configs = [
        ('008_10pct', 'WW (base=8%, sampling=10%)'),
        ('008_25pct', 'WW (base=8%, sampling=25%)'),
        ('008_50pct', 'WW (base=8%, sampling=50%)'),
        ('008_100pct', 'WW (base=8%, sampling=100%)'),
        ('016_10pct', 'WW (base=16%, sampling=10%)'),
        ('016_25pct', 'WW (base=16%, sampling=25%)'),
        ('016_50pct', 'WW (base=16%, sampling=50%)'),
        ('016_100pct', 'WW (base=16%, sampling=100%)')
    ]
    
    for config_name, config_label in ww_configs:
        print(f"\n" + "="*50)
        print(config_label)
        print("="*50)
        
        time_col = f'WW_{config_name}_mean_detection_time'
        samples_col = f'WW_{config_name}_detection_times'
        cases_col = f'WW_{config_name}_mean_local_cases'
        cases_samples_col = f'WW_{config_name}_local_cases_samples'
        
        ww_time_simple = simple_mean_ci(subset, time_col)
        ww_time_hierarchical = hierarchical_bootstrap_ci(subset, samples_col)
        print(f"Detection Time - Simple: {ww_time_simple['mean']:.1f}: ({ww_time_simple['ci_lower']:.1f}, {ww_time_simple['ci_upper']:.1f})")
        print(f"Detection Time - Hierarchical: {ww_time_hierarchical['mean']:.1f}: ({ww_time_hierarchical['ci_lower']:.1f}, {ww_time_hierarchical['ci_upper']:.1f})")
        
        ww_cases_simple = simple_mean_ci(subset, cases_col)
        ww_cases_hierarchical = hierarchical_bootstrap_ci(subset, cases_samples_col)
        print(f"Local Cases - Simple: {ww_cases_simple['mean']:.1f}: ({ww_cases_simple['ci_lower']:.1f}, {ww_cases_simple['ci_upper']:.1f})")
        print(f"Local Cases - Hierarchical: {ww_cases_hierarchical['mean']:.1f}: ({ww_cases_hierarchical['ci_lower']:.1f}, {ww_cases_hierarchical['ci_upper']:.1f})")
        
        result_dict.update({
            f'WW_{config_name}_time_simple_mean': ww_time_simple['mean'],
            f'WW_{config_name}_time_simple_ci_lower': ww_time_simple['ci_lower'],
            f'WW_{config_name}_time_simple_ci_upper': ww_time_simple['ci_upper'],
            f'WW_{config_name}_time_hierarchical_mean': ww_time_hierarchical['mean'],
            f'WW_{config_name}_time_hierarchical_pi_lower': ww_time_hierarchical['ci_lower'],
            f'WW_{config_name}_time_hierarchical_pi_upper': ww_time_hierarchical['ci_upper'],
            f'WW_{config_name}_cases_simple_mean': ww_cases_simple['mean'],
            f'WW_{config_name}_cases_simple_ci_lower': ww_cases_simple['ci_lower'],
            f'WW_{config_name}_cases_simple_ci_upper': ww_cases_simple['ci_upper'],
            f'WW_{config_name}_cases_hierarchical_mean': ww_cases_hierarchical['mean'],
            f'WW_{config_name}_cases_hierarchical_pi_lower': ww_cases_hierarchical['ci_lower'],
            f'WW_{config_name}_cases_hierarchical_pi_upper': ww_cases_hierarchical['ci_upper']
        })
    
    summary_data.append(result_dict)

print(f"\n{'='*100}")
print("ANALYSIS COMPLETE")
print(f"{'='*100}")

# Save comparison
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("/Users/reddy/AWW_and_ICU/pgfgleam/pgfgleam/datasets/comparison_simple_vs_hierarchical_CIs.csv", index=False)
print(f"\n✓ Full comparison saved to comparison_simple_vs_hierarchical_CIs.csv")
print(f"   Columns: {len(summary_df.columns)}")
print(f"   Rows: {len(summary_df)}")

Parsing ICU_detection_times...
Parsing ICU_local_cases_samples...
Parsing WW_008_10pct_detection_times...
Parsing WW_008_10pct_local_cases_samples...
Parsing WW_008_25pct_detection_times...
Parsing WW_008_25pct_local_cases_samples...
Parsing WW_008_50pct_detection_times...
Parsing WW_008_50pct_local_cases_samples...
Parsing WW_008_100pct_detection_times...
Parsing WW_008_100pct_local_cases_samples...
Parsing WW_016_10pct_detection_times...
Parsing WW_016_10pct_local_cases_samples...
Parsing WW_016_25pct_detection_times...
Parsing WW_016_25pct_local_cases_samples...
Parsing WW_016_50pct_detection_times...
Parsing WW_016_50pct_local_cases_samples...
Parsing WW_016_100pct_detection_times...
Parsing WW_016_100pct_local_cases_samples...
Loaded 3534 results

HIERARCHICAL BOOTSTRAP CIs vs SIMPLE CIs (8 WW CONFIGS + ICU)


Processing combinations:   0%|          | 0/16 [00:00<?, ?it/s]


----------------------------------------------------------------------------------------------------
R0 = 1.5, Generation Time = 4.0 days
Number of countries: 222
----------------------------------------------------------------------------------------------------

ICU
Detection Time - Simple: 83.8: (80.9, 86.7)
Detection Time - Hierarchical: 83.8: (31.0, 134.2)
Local Cases - Simple: 78.0: (76.6, 79.5)
Local Cases - Hierarchical: 77.7: (1.0, 354.0)

WW (base=8%, sampling=10%)
Detection Time - Simple: 88.8: (86.1, 91.6)
Detection Time - Hierarchical: 88.9: (40.0, 132.0)
Local Cases - Simple: 123.6: (121.7, 125.5)
Local Cases - Hierarchical: 123.6: (1.0, 462.0)

WW (base=8%, sampling=25%)
Detection Time - Simple: 80.5: (77.7, 83.3)
Detection Time - Hierarchical: 80.4: (33.0, 124.0)
Local Cases - Simple: 50.0: (49.1, 50.8)
Local Cases - Hierarchical: 50.1: (0.0, 197.0)

WW (base=8%, sampling=50%)
Detection Time - Simple: 67.9: (65.0, 70.7)
Detection Time - Hierarchical: 67.8: (20.0, 112.0

Processing combinations:   6%|▋         | 1/16 [00:17<04:21, 17.42s/it]

Local Cases - Simple: 6.3: (6.1, 6.4)
Local Cases - Hierarchical: 6.3: (0.0, 33.0)

----------------------------------------------------------------------------------------------------
R0 = 1.5, Generation Time = 6.0 days
Number of countries: 221
----------------------------------------------------------------------------------------------------

ICU
Detection Time - Simple: 110.8: (106.9, 114.7)
Detection Time - Hierarchical: 110.6: (38.8, 175.3)
Local Cases - Simple: 48.4: (47.5, 49.2)
Local Cases - Hierarchical: 48.5: (1.0, 169.0)

WW (base=8%, sampling=10%)
Detection Time - Simple: 117.0: (113.1, 120.9)
Detection Time - Hierarchical: 117.1: (51.0, 177.0)
Local Cases - Simple: 64.8: (63.4, 66.2)
Local Cases - Hierarchical: 65.0: (0.0, 183.0)

WW (base=8%, sampling=25%)
Detection Time - Simple: 110.3: (106.4, 114.2)
Detection Time - Hierarchical: 110.3: (42.0, 171.0)
Local Cases - Simple: 44.2: (43.5, 44.8)
Local Cases - Hierarchical: 43.9: (0.0, 150.0)

WW (base=8%, sampling=50%)
De

...

Local Cases - Simple: 12.6: (12.4, 12.9)
Local Cases - Hierarchical: 12.5: (0.0, 61.0)

----------------------------------------------------------------------------------------------------
R0 = 3.0, Generation Time = 8.0 days
Number of countries: 222
----------------------------------------------------------------------------------------------------

ICU
Detection Time - Simple: 85.8: (84.0, 87.6)
Detection Time - Hierarchical: 85.9: (41.1, 120.9)
Local Cases - Simple: 2244.3: (2212.2, 2276.5)
Local Cases - Hierarchical: 2249.5: (6.0, 6215.0)

WW (base=8%, sampling=10%)
Detection Time - Simple: 62.6: (60.8, 64.3)
Detection Time - Hierarchical: 62.6: (32.0, 90.0)
Local Cases - Simple: 202.4: (198.8, 206.0)
Local Cases - Hierarchical: 201.5: (2.0, 788.0)

WW (base=8%, sampling=25%)
Detection Time - Simple: 57.5: (55.8, 59.3)
Detection Time - Hierarchical: 57.5: (27.0, 85.0)
Local Cases - Simple: 81.0: (79.5, 82.5)
Local Cases - Hierarchical: 81.4: (0.0, 321.0)

WW (base=8%, sampling=50%)

Processing combinations:  94%|█████████▍| 15/16 [04:04<00:16, 16.42s/it]

Local Cases - Simple: 9.9: (9.6, 10.1)
Local Cases - Hierarchical: 9.9: (0.0, 48.0)

----------------------------------------------------------------------------------------------------
R0 = 3.0, Generation Time = 10.0 days
Number of countries: 222
----------------------------------------------------------------------------------------------------

ICU
Detection Time - Simple: 92.2: (90.1, 94.4)
Detection Time - Hierarchical: 92.2: (42.3, 132.8)
Local Cases - Simple: 1338.0: (1318.5, 1357.6)
Local Cases - Hierarchical: 1341.5: (4.0, 3744.0)

WW (base=8%, sampling=10%)
Detection Time - Simple: 73.2: (71.1, 75.2)
Detection Time - Hierarchical: 73.2: (38.0, 105.0)
Local Cases - Simple: 225.3: (220.9, 229.6)
Local Cases - Hierarchical: 225.4: (2.0, 866.0)

WW (base=8%, sampling=25%)
Detection Time - Simple: 67.0: (65.0, 69.1)
Detection Time - Hierarchical: 67.1: (32.0, 99.0)
Local Cases - Simple: 89.6: (87.9, 91.2)
Local Cases - Hierarchical: 89.0: (0.0, 345.0)

WW (base=8%, sampling=50%)


Processing combinations: 100%|██████████| 16/16 [04:21<00:00, 16.32s/it]

Local Cases - Simple: 10.7: (10.5, 10.9)
Local Cases - Hierarchical: 10.8: (0.0, 52.0)

ANALYSIS COMPLETE

✓ Full comparison saved to comparison_simple_vs_hierarchical_CIs.csv
   Columns: 111
   Rows: 16


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the comparison results
comparison = pd.read_csv("/Users/reddy/AWW_and_ICU/pgfgleam/pgfgleam/datasets/comparison_simple_vs_hierarchical_CIs.csv")

# Define only the 16% base configurations with 25% and 50% sampling
ww_configs = [
    ('016_25pct', '16% base, 25% sampling', 0.040),
    ('016_50pct', '16% base, 50% sampling', 0.080),
]

def ci_to_se(ci_lower, ci_upper):
    """Convert 95% CI bounds to standard error."""
    return (ci_upper - ci_lower) / 3.92

def se_to_ci(mean, se):
    """Convert mean and SE to 95% CI bounds."""
    return mean - 1.96 * se, mean + 1.96 * se

# Compute differences and ratios with propagated CIs
for config_name, config_label, eff_pdet in ww_configs:
    ww_prefix = f'WW_{config_name}'

    # --- Means ---
    comparison[f'ICU_minus_{ww_prefix}_time_simple'] = (
        comparison['ICU_time_simple_mean'] - comparison[f'{ww_prefix}_time_simple_mean']
    )
    comparison[f'ICU_minus_{ww_prefix}_cases_simple'] = (
        comparison['ICU_cases_simple_mean'] - comparison[f'{ww_prefix}_cases_simple_mean']
    )
    comparison[f'ICU_div_{ww_prefix}_cases_simple'] = (
        comparison['ICU_cases_simple_mean'] / comparison[f'{ww_prefix}_cases_simple_mean']
    )

    # --- Standard errors from stored CIs ---
    se_icu_time   = ci_to_se(comparison['ICU_time_simple_ci_lower'],   comparison['ICU_time_simple_ci_upper'])
    se_ww_time    = ci_to_se(comparison[f'{ww_prefix}_time_simple_ci_lower'],  comparison[f'{ww_prefix}_time_simple_ci_upper'])
    se_icu_cases  = ci_to_se(comparison['ICU_cases_simple_ci_lower'],  comparison['ICU_cases_simple_ci_upper'])
    se_ww_cases   = ci_to_se(comparison[f'{ww_prefix}_cases_simple_ci_lower'], comparison[f'{ww_prefix}_cases_simple_ci_upper'])

    # --- Propagated CIs for differences ---
    se_time_diff  = np.sqrt(se_icu_time**2  + se_ww_time**2)
    se_cases_diff = np.sqrt(se_icu_cases**2 + se_ww_cases**2)

    comparison[f'ICU_minus_{ww_prefix}_time_ci_lower'],  comparison[f'ICU_minus_{ww_prefix}_time_ci_upper']  = \
        se_to_ci(comparison[f'ICU_minus_{ww_prefix}_time_simple'],  se_time_diff)
    comparison[f'ICU_minus_{ww_prefix}_cases_ci_lower'], comparison[f'ICU_minus_{ww_prefix}_cases_ci_upper'] = \
        se_to_ci(comparison[f'ICU_minus_{ww_prefix}_cases_simple'], se_cases_diff)

    # --- Propagated CIs for ratio via delta method: SE(A/B) ≈ (A/B)*sqrt((SE_A/A)^2 + (SE_B/B)^2) ---
    mu_icu   = comparison['ICU_cases_simple_mean']
    mu_ww    = comparison[f'{ww_prefix}_cases_simple_mean']
    ratio    = comparison[f'ICU_div_{ww_prefix}_cases_simple']
    se_ratio = ratio * np.sqrt((se_icu_cases / mu_icu)**2 + (se_ww_cases / mu_ww)**2)

    comparison[f'ICU_div_{ww_prefix}_cases_ci_lower'], comparison[f'ICU_div_{ww_prefix}_cases_ci_upper'] = \
        se_to_ci(ratio, se_ratio)


# ── Detailed results ──────────────────────────────────────────────────────────
print("\n" + "="*110)
print("DETAILED RESULTS BY PARAMETER COMBINATION")
print("="*110)

for _, row in comparison.iterrows():
    print(f"\nR0 = {row['R0']}, Generation Time = {row['gen_time']} days (n={row['n_countries']} countries)")
    print("-" * 110)

    for config_name, config_label, eff_pdet in ww_configs:
        ww_prefix = f'WW_{config_name}'

        time_diff  = row[f'ICU_minus_{ww_prefix}_time_simple']
        time_lo    = row[f'ICU_minus_{ww_prefix}_time_ci_lower']
        time_hi    = row[f'ICU_minus_{ww_prefix}_time_ci_upper']

        cases_diff = row[f'ICU_minus_{ww_prefix}_cases_simple']
        cases_lo   = row[f'ICU_minus_{ww_prefix}_cases_ci_lower']
        cases_hi   = row[f'ICU_minus_{ww_prefix}_cases_ci_upper']

        ratio      = row[f'ICU_div_{ww_prefix}_cases_simple']
        ratio_lo   = row[f'ICU_div_{ww_prefix}_cases_ci_lower']
        ratio_hi   = row[f'ICU_div_{ww_prefix}_cases_ci_upper']

        print(f"  {config_label} (effective p_det={eff_pdet}):")
        print(f"    Detection Time Difference  (ICU - WW): {time_diff:+.2f} days  "
              f"[95% CI: {time_lo:+.2f}, {time_hi:+.2f}]")
        print(f"    Local Cases Difference     (ICU - WW): {cases_diff:+.1f} cases "
              f"[95% CI: {cases_lo:+.1f}, {cases_hi:+.1f}]")
        print(f"    Local Cases Ratio          (ICU / WW): {ratio:.2f}x              "
              f"[95% CI: {ratio_lo:.2f}x, {ratio_hi:.2f}x]")


# ── Summary statistics ────────────────────────────────────────────────────────
print("\n" + "="*110)
print("SUMMARY STATISTICS ACROSS ALL R0/GENERATION TIME COMBINATIONS")
print("="*110)

for config_name, config_label, eff_pdet in ww_configs:
    ww_prefix = f'WW_{config_name}'

    time_diff_col  = f'ICU_minus_{ww_prefix}_time_simple'
    cases_diff_col = f'ICU_minus_{ww_prefix}_cases_simple'
    cases_ratio_col = f'ICU_div_{ww_prefix}_cases_simple'

    print(f"\n{config_label} (effective p_det={eff_pdet}):")

    print(f"  Detection Time Difference (ICU - WW):")
    print(f"    Range:   [{comparison[time_diff_col].min():+.2f}, {comparison[time_diff_col].max():+.2f}] days")
    print(f"    Mean ± SD: {comparison[time_diff_col].mean():+.2f} ± {comparison[time_diff_col].std():.2f} days")

    print(f"  Local Cases Difference (ICU - WW):")
    print(f"    Range:   [{comparison[cases_diff_col].min():+.1f}, {comparison[cases_diff_col].max():+.1f}] cases")
    print(f"    Mean ± SD: {comparison[cases_diff_col].mean():+.1f} ± {comparison[cases_diff_col].std():.1f} cases")

    print(f"  Local Cases Ratio (ICU / WW):")
    print(f"    Range:   [{comparison[cases_ratio_col].min():.2f}x, {comparison[cases_ratio_col].max():.2f}x]")
    print(f"    Mean ± SD: {comparison[cases_ratio_col].mean():.2f}x ± {comparison[cases_ratio_col].std():.2f}")


DETAILED RESULTS BY PARAMETER COMBINATION

R0 = 1.5, Generation Time = 4.0 days (n=222.0 countries)
--------------------------------------------------------------------------------------------------------------
  16% base, 25% sampling (effective p_det=0.04):
    Detection Time Difference  (ICU - WW): +15.95 days  [95% CI: +11.91, +19.99]
    Local Cases Difference     (ICU - WW): +65.4 cases [95% CI: +64.0, +66.8]
    Local Cases Ratio          (ICU / WW): 6.17x              [95% CI: 6.02x, 6.33x]
  16% base, 50% sampling (effective p_det=0.08):
    Detection Time Difference  (ICU - WW): +21.77 days  [95% CI: +17.73, +25.81]
    Local Cases Difference     (ICU - WW): +71.5 cases [95% CI: +70.1, +72.9]
    Local Cases Ratio          (ICU / WW): 11.98x              [95% CI: 11.66x, 12.29x]

R0 = 1.5, Generation Time = 6.0 days (n=221.0 countries)
--------------------------------------------------------------------------------------------------------------
  16% base, 25% sampling (effe

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the comparison results
comparison = pd.read_csv("/Users/reddy/AWW_and_ICU/pgfgleam/pgfgleam/datasets/comparison_simple_vs_hierarchical_CIs.csv")

# Define only the 16% base configurations with 25% and 50% sampling
ww_configs = [
    ('016_25pct', '16% base, 25% sampling', 0.040),
    ('016_50pct', '16% base, 50% sampling', 0.080),
]

# Compute differences and ratios for selected configurations
for config_name, config_label, eff_pdet in ww_configs:
    time_diff_col = f'ICU_minus_WW{config_name}_time_simple'
    cases_diff_col = f'ICU_minus_WW{config_name}_cases_simple'
    cases_ratio_col = f'ICU_div_WW{config_name}_cases_simple'
    
    comparison[time_diff_col] = comparison['ICU_time_simple_mean'] - comparison[f'WW_{config_name}_time_simple_mean']
    comparison[cases_diff_col] = comparison['ICU_cases_simple_mean'] - comparison[f'WW_{config_name}_cases_simple_mean']
    comparison[cases_ratio_col] = comparison['ICU_cases_simple_mean'] / comparison[f'WW_{config_name}_cases_simple_mean']

# Print detailed results for each R0/gen_time combination
print("\n" + "="*100)
print("DETAILED RESULTS BY PARAMETER COMBINATION")
print("="*100)

for _, row in comparison.iterrows():
    print(f"\nR0 = {row['R0']}, Generation Time = {row['gen_time']} days (n={row['n_countries']} countries)")
    print("-" * 100)
    
    for config_name, config_label, eff_pdet in ww_configs:
        time_diff = row[f'ICU_minus_WW{config_name}_time_simple']
        cases_diff = row[f'ICU_minus_WW{config_name}_cases_simple']
        cases_ratio = row[f'ICU_div_WW{config_name}_cases_simple']
        
        print(f"  {config_label} (effective p_det={eff_pdet}):")
        print(f"    Detection Time Difference (ICU - WW): {time_diff:+.2f} days")
        print(f"    Local Cases Difference (ICU - WW): {cases_diff:+.1f} cases")
        print(f"    Local Cases Ratio (ICU / WW): {cases_ratio:.2f}x")

# Print summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS ACROSS ALL R0/GENERATION TIME COMBINATIONS")
print("="*100)

for config_name, config_label, eff_pdet in ww_configs:
    time_diff_col = f'ICU_minus_WW{config_name}_time_simple'
    cases_diff_col = f'ICU_minus_WW{config_name}_cases_simple'
    cases_ratio_col = f'ICU_div_WW{config_name}_cases_simple'
    
    print(f"\n{config_label} (effective p_det={eff_pdet}):")
    print(f"  Detection Time Difference (ICU - WW):")
    print(f"    Range: [{comparison[time_diff_col].min():+.2f}, {comparison[time_diff_col].max():+.2f}] days")
    print(f"    Mean ± SD: {comparison[time_diff_col].mean():+.2f} ± {comparison[time_diff_col].std():.2f} days")
    
    print(f"  Local Cases Difference (ICU - WW):")
    print(f"    Range: [{comparison[cases_diff_col].min():+.1f}, {comparison[cases_diff_col].max():+.1f}] cases")
    print(f"    Mean ± SD: {comparison[cases_diff_col].mean():+.1f} ± {comparison[cases_diff_col].std():.1f} cases")
    
    print(f"  Local Cases Ratio (ICU / WW):")
    print(f"    Range: [{comparison[cases_ratio_col].min():.2f}x, {comparison[cases_ratio_col].max():.2f}x]")
    print(f"    Mean ± SD: {comparison[cases_ratio_col].mean():.2f}x ± {comparison[cases_ratio_col].std():.2f}")



DETAILED RESULTS BY PARAMETER COMBINATION

R0 = 1.5, Generation Time = 4.0 days (n=222.0 countries)
----------------------------------------------------------------------------------------------------
  16% base, 25% sampling (effective p_det=0.04):
    Detection Time Difference (ICU - WW): +15.95 days
    Local Cases Difference (ICU - WW): +65.4 cases
    Local Cases Ratio (ICU / WW): 6.17x
  16% base, 50% sampling (effective p_det=0.08):
    Detection Time Difference (ICU - WW): +21.77 days
    Local Cases Difference (ICU - WW): +71.5 cases
    Local Cases Ratio (ICU / WW): 11.98x

R0 = 1.5, Generation Time = 6.0 days (n=221.0 countries)
----------------------------------------------------------------------------------------------------
  16% base, 25% sampling (effective p_det=0.04):
    Detection Time Difference (ICU - WW): +17.46 days
    Local Cases Difference (ICU - WW): +35.5 cases
    Local Cases Ratio (ICU / WW): 3.75x
  16% base, 50% sampling (effective p_det=0.08):
    Det